# 02 - Définition de l'alerte et baseline NDWI

1. **Définir l'alerte** : une image mérite une alerte si une part suffisante de sa surface est inondée
   (eau annotée **hors** eau permanente).
2. **Découper par pays** : entraînement, validation et test sur des événements différents.
3. **Baseline simple et explicable** : indice d'eau NDWI + deux seuils, réglés sur l'entraînement.
4. **Évaluer** : précision, rappel, taux de fausses alertes, temps de calcul, cas d'échec.

Exécution : Google Colab, *Exécution > Tout exécuter* (environ 3 minutes).

In [ ]:
import os, subprocess, sys

# Récupère le code du projet (dossier src/) depuis GitHub
if not os.path.isdir("one-look"):
    subprocess.run(["git", "clone", "-q", "https://github.com/lemembre05/one-look.git"], check=True)
else:
    subprocess.run(["git", "-C", "one-look", "pull", "-q"], check=True)
sys.path.insert(0, "one-look")

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "rasterio"], check=True)

from src import donnees
noms = donnees.telecharger("data")
print(len(noms), "images")

## 1. Mesures par image

Pour chaque image on calcule :
- la **part inondée réelle** (eau annotée hors eau permanente), qui sert de vérité ;
- la **part de nuages** (pixels sans annotation) ;
- l'**histogramme du NDWI** sur les pixels hors eau permanente. Il permet ensuite de tester
  n'importe quel seuil NDWI sans relire les images.

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from src.metriques import scores_alerte

BORDS = np.linspace(-1, 1, 201)          # 200 classes de NDWI de largeur 0,01

lignes, histos = [], []
for nom in noms:
    s2, label, perm = donnees.lire_image(nom, "data")
    vu = s2.max(axis=0) > 0
    utiles = donnees.pixels_utiles(s2, perm)
    h, _ = np.histogram(np.clip(donnees.ndwi(s2)[utiles], -1, 1), bins=BORDS)
    histos.append(h)
    lignes.append({"image": nom,
                   "pays": donnees.pays(nom),
                   "partie": donnees.partie(nom),
                   "part_inondee": donnees.part_inondee_vraie(label, perm),
                   "part_nuages": float((label < 0).mean()),
                   "pixels_vus": int(vu.sum())})

df = pd.DataFrame(lignes)
H = np.array(histos)
print(df[["part_inondee", "part_nuages"]].describe().round(3))

## 2. Définition de l'alerte

Combien d'images déclenchent une alerte selon le seuil choisi ?
Un seuil trop bas alerte pour quelques flaques ; un seuil trop haut rate les inondations débutantes.

In [ ]:
seuils = [0.01, 0.02, 0.05, 0.10, 0.20]
tab = pd.DataFrame({f"> {int(s*100)} %": df.groupby("partie")["part_inondee"].apply(lambda x: int((x >= s).sum()))
                    for s in seuils})
tab["total images"] = df.groupby("partie").size()
print(tab)

In [ ]:
# >>> Paramètre à justifier : part minimale de surface inondée pour déclencher une alerte
SEUIL_ALERTE = 0.05

df["alerte_vraie"] = df["part_inondee"] >= SEUIL_ALERTE
print(f"Alerte si au moins {SEUIL_ALERTE:.0%} de l'image est inondée (hors eau permanente)\n")
print(df.groupby(["partie", "pays"])["alerte_vraie"].agg(images="size", alertes="sum"))

## 3. Baseline : NDWI + deux seuils

Règle, appliquée à chaque image :
1. un pixel est de l'**eau** si NDWI > `t_ndwi` (et s'il n'est pas de l'eau permanente) ;
2. l'image déclenche une **alerte** si la part de pixels d'eau dépasse `t_alerte`.

Les deux seuils sont choisis sur l'**entraînement uniquement** (meilleur F1).
La validation et le test ne servent qu'à mesurer.

In [ ]:
def part_eau_predite(t_ndwi):
    """Part de pixels avec NDWI > t_ndwi (hors eau permanente), pour chaque image."""
    k = np.searchsorted(BORDS, t_ndwi - 1e-9)      # première classe qui commence à t_ndwi
    return H[:, k:].sum(axis=1) / np.maximum(df["pixels_vus"].to_numpy(), 1)

train = (df["partie"] == "train").to_numpy()
essais = []
for t_ndwi in np.round(np.arange(-0.30, 0.61, 0.02), 2):
    part = part_eau_predite(t_ndwi)
    for t_alerte in [0.005, 0.01, 0.02, 0.03, 0.05, 0.08, 0.10, 0.15]:
        s = scores_alerte(df["alerte_vraie"][train], part[train] >= t_alerte)
        essais.append({"t_ndwi": t_ndwi, "t_alerte": t_alerte, **s})

essais = pd.DataFrame(essais)
# En cas d'égalité : moins de fausses alertes, puis t_alerte le plus proche de la définition de l'alerte
essais["ecart"] = (essais["t_alerte"] - SEUIL_ALERTE).abs()
essais = essais.sort_values(["f1", "taux_fausse_alerte", "ecart"], ascending=[False, True, True])
print(essais.head(8)[["t_ndwi", "t_alerte", "precision", "rappel", "f1", "taux_fausse_alerte"]].to_string(index=False))

T_NDWI, T_ALERTE = essais.iloc[0]["t_ndwi"], essais.iloc[0]["t_alerte"]
print(f"\nSeuils retenus : t_ndwi = {T_NDWI}, t_alerte = {T_ALERTE}")

## 4. Évaluation

In [ ]:
df["part_eau_predite"] = part_eau_predite(T_NDWI)
df["alerte_predite"] = df["part_eau_predite"] >= T_ALERTE

resultats = pd.DataFrame({p: scores_alerte(g["alerte_vraie"], g["alerte_predite"])
                          for p, g in df.groupby("partie")}).T.loc[["train", "validation", "test"]]
print(resultats.to_string())

print("\nTest, par pays :")
print(pd.DataFrame({p: scores_alerte(g["alerte_vraie"], g["alerte_predite"])
                    for p, g in df[df["partie"] == "test"].groupby("pays")}).T.to_string())

## 5. Embarquabilité

La baseline ne contient que deux nombres. On mesure le temps de calcul sur une image complète
(512 x 512 pixels, 2 bandes utilisées), sur le processeur de Colab.

In [ ]:
import time

def baseline(s2, perm):
    eau = (donnees.ndwi(s2) > T_NDWI) & donnees.pixels_utiles(s2, perm)
    return eau.sum() / max((s2.max(axis=0) > 0).sum(), 1) >= T_ALERTE

images = [donnees.lire_image(n, "data") for n in noms[:20]]
debut = time.perf_counter()
for s2, _, perm in images:
    baseline(s2, perm)
ms = (time.perf_counter() - debut) / len(images) * 1000
print(f"Temps moyen : {ms:.1f} ms par image | taille du modèle : 2 paramètres")

## 6. Cas d'échec (validation et test)

Fausses alertes et inondations manquées, pour comprendre les limites de la règle.
De gauche à droite : image, pixels détectés comme eau, inondation annotée.

In [ ]:
eval_ = df[df["partie"] != "train"]
cas = pd.concat([eval_[eval_["alerte_predite"] & ~eval_["alerte_vraie"]].head(3).assign(type="Fausse alerte"),
                 eval_[~eval_["alerte_predite"] & eval_["alerte_vraie"]].head(3).assign(type="Manquée")])

for _, c in cas.iterrows():
    s2, label, perm = donnees.lire_image(c["image"], "data")
    detecte = (donnees.ndwi(s2) > T_NDWI) & donnees.pixels_utiles(s2, perm)
    vrai = (label == 1) & ~perm
    fig, ax = plt.subplots(1, 3, figsize=(12, 4))
    ax[0].imshow(donnees.rgb(s2)); ax[0].set_title(f"{c['type']} : {c['image']}")
    ax[1].imshow(detecte, cmap="Blues"); ax[1].set_title(f"Détecté : {c['part_eau_predite']:.1%}")
    ax[2].imshow(vrai, cmap="Blues");    ax[2].set_title(f"Annoté : {c['part_inondee']:.1%} | nuages {c['part_nuages']:.0%}")
    for a in ax: a.axis("off")
    plt.show()

if cas.empty:
    print("Aucun cas d'échec en validation et test.")

## À retenir

- La vérité est définie par **un seul paramètre**, `SEUIL_ALERTE`, à justifier dans le rapport.
- La baseline est **entièrement explicable** : un indice physique (l'eau absorbe le proche infrarouge)
  et deux seuils.
- Les cas d'échec montrent ce que le modèle de deep learning devra améliorer.